In [6]:
PATHS = ["./datasets/HuggingFaceM4/ChartQA/data/train-00000-of-00003-49492f364babfa44.parquet", 
        "./datasets/HuggingFaceM4/ChartQA/data/train-00001-of-00003-7302bae5e425bbc7.parquet",
        "./datasets/HuggingFaceM4/ChartQA/data/train-00002-of-00003-194c9400785577a2.parquet",
        "./datasets/HuggingFaceM4/ChartQA/data/val-00000-of-00001-0f11003c77497969.parquet"
        
        ]

In [ ]:
import pandas as pd

# Load only the first few rows to inspect the structure

df_val = pd.read_parquet(PATHS[-1], engine="pyarrow")
df_val.size




(1920, 4)
['image', 'query', 'label', 'human_or_machine']
{'image': {'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x02\x80\x00\x00\x01\x82\x08\x02\x00\x00\x00\xff\x8f\xe5\xe2\x00\x01\x00\x00IDATx\x9c\xec\xddux\x14\xd7\xde\x07\xf0\xef\xcc\xfan\xdc\xdd\xdd\x03\x04\x0b\xee\xeen-\xd4[*\x94\x16(\xf5\x02u\x83Roii\x91R\xdc\xdd\x9d$\x10!\xee\xee\xbe\xbe;s\xde?6\t\t\xd0\x16n\x81\xf4m\xcf\xe7yn/\x99\x9d=sfvf~st\x18B\x08(\x8a\xa2(\x8az\xb0\x84\x00\n\n\njkk;;\'\x14EQ\x14\xf5_\x11\x19\x19)\x04@\x08\xa1\xe5`\x8a\xa2(\x8az\x90\x18\x1az)\x8a\xa2(\xea\xc1c;;\x03\x14EQ\x14\xf5_D\x030EQ\x14Eu\x02\x1a\x80)\x8a\xa2(\xaa\x13\xd0\x00LQ\x14EQ\x9d\x80\x06`\x8a\xa2(\x8a\xea\x044\x00S\x14EQT\'\xa0\x01\x98\xa2(\x8a\xa2:\x01\r\xc0\x14EQ\x14\xd5\th\x00\xa6(\x8a\xa2\xa8N@\x030EQ\x14Eu\x02\x1a\x80)\x8a\xa2(\xaa\x13\xd0\x00LQ\x14EQ\x9d@x\xd3\xdf\xe59\xd7.%f\x1b\x88\xd0\'\xack\x97`O\x1a\x9f\xff\x92\xaa2\xfb\xc0\xe9k\x04l\xb7!\xa3|l\xe4\xf7<\xfd\x94\xf3\x07\xd3K\x9bE\n\xc7Q\xa3\xfbK\xfeF:\xfa\xc6\xb2C\xc7\xcei9

In [17]:
STRATA_COL = "human_or_machine"

def make_sample_df_preserved( df: pd.DataFrame, n: int, seed: int = 42,) -> pd.DataFrame:
    
    """Draw a stratified sample of size `n`, preserving all original columns."""
    
    # Proportional target size per stratum
    shares = df[STRATA_COL].value_counts(normalize=True) * n
    per_group = shares.astype(int)

    # Largest-remainder allocation of the leftover slots
    leftover = n - per_group.sum()
    by_remainder = (shares - per_group).sort_values(ascending=False)
    for group in by_remainder.index[:leftover]:
        per_group[group] += 1

    # Stratified sampling using the allocated counts
    parts = [
        df[df[STRATA_COL] == group].sample(n=int(k), random_state=seed)
        for group, k in per_group.items()
    ]

    return pd.concat(parts).sort_index().reset_index(drop=True)

In [18]:
df_sample = make_sample_df_preserved(df_val, 100, 24)
df_sample.size

print(df_sample.shape)
print(df_sample.columns.tolist())
print(df_sample.iloc[0].to_dict())

(100, 4)
['image', 'query', 'label', 'human_or_machine']
{'image': {'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01\x9f\x00\x00\x01\xd3\x08\x02\x00\x00\x00w\x1a\x82\xd4\x00\x00i\x19IDATx\x9c\xed\xbd\x7fTU\xd7\xb9\xf7;\xf1G\xa2\x82\xc2\x16\r\x88\xa8to0\xb6\xe6V\x9a\x03z\x12\xb71\xc9\xa5`\x8e\xc90\xe6\xcd[\x81$\x1en\xc7{\xed\x00\xda3r\xbcV\xcf{\xaa\'Xs\xda\x0316\xb7\xad0\x9aqGJ\xc4\x17Ho\x8e\xe8Imd\x87k\x12\xd9\xc6\n\x9cDz4%\xc0\xda%JP\x12a\x83\x8011\xba\xef\x1f\xdf\xf0t:\xd7\xda\x8b\xcd\x86\xcd\x8f\xe5\xf3\x19\x8ed1\xf7Zs=\xf3\xd7\xb3\xe6\xaf\xe7\x99a>\x9fO0\x0c\xc3X\x8e)\xe3-\x00\xc30Lh\xe0\xbe\x1b\xc30\xd6\xc3\xe7\xf3\x85\xb1vc\x18\xc6\x92\xf0\xc8\x94a\x18k\xc2\xda\x8da\x18k\xc2\xda\x8da\x18k\xc2\xda\x8da\x18k\xc2\xda\x8da\x18k\xc2\xda\x8da\x18k\xc2\xda\x8da\x18k\xc2\xda\x8da\x18k\xc2\xda\x8da\x18k\xc2\xda\x8da\x18k\xc2\xda\x8da\x18k\xc2\xda\x8da\x18k\xc2\xda\x8da\x18k\xf2\xb5vKLL\x0c\xd3\xb1n\xdd\xba@\xa2(**\xc2\xfdrTyyy\x81<K\xef\xad\xac\xac4\x8cV\x96\x01w\x16\x15\x15\x05

In [10]:
import io
import matplotlib.pyplot as plt
from PIL import Image

def decode_chart_image(row, image_col='image'):
    val = row[image_col]
    if isinstance(val, dict) and 'bytes' in val:
        return Image.open(io.BytesIO(val['bytes']))
    elif isinstance(val, bytes):
        return Image.open(io.BytesIO(val))
    elif isinstance(val, str):
        return Image.open(val)
    else:
        raise ValueError("Unsupported image column format. Ensure it contains bytes or paths.")

In [19]:
import time
import torch
import pandas as pd




device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

print("device:", device)


device: mps


In [20]:
from transformers import (
    DonutProcessor, VisionEncoderDecoderModel,
)


# --- UniChart ---
uc_name = "ahmed-masry/unichart-base-960"
uc_proc = DonutProcessor.from_pretrained(uc_name)
uc_model = VisionEncoderDecoderModel.from_pretrained(uc_name).to(device).eval()

def unichart_table(img):
    prompt = "<extract_data_table> <s_answer>"
    decoder_ids = uc_proc.tokenizer(prompt, add_special_tokens=False, return_tensors="pt").input_ids
    pixel_values = uc_proc(img, return_tensors="pt").pixel_values
    with torch.no_grad():
        out = uc_model.generate(
            pixel_values.to(device),
            decoder_input_ids=decoder_ids.to(device),
            max_length=uc_model.decoder.config.max_position_embeddings,
            pad_token_id=uc_proc.tokenizer.pad_token_id,
            eos_token_id=uc_proc.tokenizer.eos_token_id,
            use_cache=True,
            num_beams=4,
            return_dict_in_generate=True,
        )
    seq = uc_proc.tokenizer.batch_decode(out.sequences)[0]
    seq = seq.replace(uc_proc.tokenizer.eos_token, "").replace(uc_proc.tokenizer.pad_token, "")
    return seq.split("<s_answer>")[-1].strip()



Loading weights: 100%|██████████| 484/484 [00:00<00:00, 30204.48it/s]


In [ ]:
# --- run on the test sample ---
def label_in(text, label):
    """Crude check: is the ground-truth answer present in the extracted table?"""
    return str(label[0]).lower() in text.lower()

results = []
for _, row in df_sample.iterrows():
    img = decode_chart_image(row).convert("RGB")

    
    t0 = time.time(); uc = unichart_table(img); t_uc = time.time() - t0

    results.append({
        "query": row["query"],
        "label": row["label"],
        
        "unichart_s": round(t_uc, 2),
      
        "unichart_has_label": label_in(uc, row["label"]),
    })

df_ocr = pd.DataFrame(results)

# summary


KeyError: "['deplot_s'] not in index"

In [22]:
df_ocr.head()

,query,label,unichart_s,unichart_has_label
0,What is the max value of blue line?,[0.72],5.70,False
1,What is the total percentage of Afghan applica...,[0.34],4.31,False
2,What's the average value of all dark blue bars?,[62.75],3.79,False
3,Which two places mentioned in the chart?,"[[Madagascar, Central Europe]]",1.13,False
4,What is the sum of largest and smallest bar?,[159.85],1.95,False


In [ ]:
print(df_ocr[[ "unichart_s"]].mean().round(2).to_string())
print(df_ocr[["unichart_has_label"]].mean().round(2).to_string())

# eyeball a few
for i in range(3):
    r = df_ocr.iloc[i]
    print(f"\n=== Q: {r['query']} | label: {r['label']} ===")
   
    print("--- UniChart ---\n", r["unichart_table"])